In [ ]:
# Multi-Metric Evaluation Framework for LLM Responses in Healthcare
# Authors: Macdonald Emeshieobi

import os
import re
import json
import warnings
from typing import List, Optional
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
from numpy.linalg import norm

from dotenv import load_dotenv

import requests
from Bio import Entrez
from bs4 import BeautifulSoup

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

from rank_bm25 import BM25Okapi

import openai
from openai import OpenAI
from anthropic import Anthropic

warnings.simplefilter(action='ignore', category=FutureWarning)

load_dotenv()
OPENAI_KEY   = os.getenv("OPENAI_API_KEY")
CLAUDE_KEY   = os.getenv("ANTHROPIC_API_KEY")
DEEPSEEK_KEY = os.getenv("DEEPSEEK_API_KEY")

Entrez.email = "emeshieobim@gmail.com"

print("Imports loaded")
print(f"OpenAI key:   {'loaded' if OPENAI_KEY else 'MISSING'}")
print(f"Claude key:   {'loaded' if CLAUDE_KEY else 'MISSING'}")
print(f"DeepSeek key: {'loaded' if DEEPSEEK_KEY else 'MISSING'}")

In [ ]:
sample_queries = pd.read_csv("sample_responses.csv")
sample_queries = sample_queries.head(32)

print(f"Loaded {len(sample_queries)} queries")
print(f"Columns: {list(sample_queries.columns)}")
sample_queries.head()

In [ ]:
def get_medical_advice_chatgpt(patient_query: str, api_key: str, model: str = "gpt-3.5-turbo") -> Optional[str]:
    """Generate a medical response using ChatGPT."""
    if not patient_query or not isinstance(patient_query, str):
        raise ValueError("Patient query must be a non-empty string")

    system_message = """You are a highly knowledgeable and compassionate doctor.
1. Carefully analyze the patient's symptoms and concerns.
2. Suggest possible causes, but never give a definitive diagnosis.
3. Offer evidence-based advice for relief and care.
4. Always recommend in-person consultation with a healthcare provider.
5. Be clear, empathetic, and professional in tone.
6. Consider lifestyle and other details mentioned by the patient.
7. Avoid exaggeration or alarming language."""

    try:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user",   "content": patient_query}
            ],
            temperature=0.2,
            max_tokens=800
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"ChatGPT error: {e}")
        return None


def get_medical_advice_claude(patient_query: str, api_key: str, model: str = "claude-3-opus-20240229") -> Optional[str]:
    """Generate a medical response using Claude."""
    if not patient_query or not isinstance(patient_query, str):
        raise ValueError("Patient query must be a non-empty string")

    system_message = """You are a highly knowledgeable and compassionate doctor.
1. Carefully analyze the patient's symptoms and concerns.
2. Suggest possible causes, but never give a definitive diagnosis.
3. Offer evidence-based advice for relief and care.
4. Always recommend in-person consultation with a healthcare provider.
5. Be clear, empathetic, and professional in tone.
6. Consider lifestyle and other details mentioned by the patient.
7. Avoid exaggeration or alarming language."""

    try:
        client = Anthropic(api_key=api_key)
        response = client.messages.create(
            model=model,
            system=system_message,
            max_tokens=800,
            temperature=0.2,
            messages=[{"role": "user", "content": patient_query}]
        )
        return response.content[0].text
    except Exception as e:
        print(f"Claude error: {e}")
        return None


def get_medical_advice_deepseek(patient_query: str, api_key: str, model: str = "deepseek-chat") -> Optional[str]:
    """Generate a medical response using DeepSeek V3."""
    if not patient_query or not isinstance(patient_query, str):
        raise ValueError("Patient query must be a non-empty string")

    system_message = """You are a highly knowledgeable and compassionate doctor.
1. Carefully analyze the patient's symptoms and concerns.
2. Suggest possible causes, but never give a definitive diagnosis.
3. Offer evidence-based advice for relief and care.
4. Always recommend in-person consultation with a healthcare provider.
5. Be clear, empathetic, and professional in tone.
6. Consider lifestyle and other details mentioned by the patient.
7. Avoid exaggeration or alarming language."""

    try:
        url = "https://api.deepseek.com/v1/chat/completions"
        headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user",   "content": patient_query}
            ],
            "temperature": 0.2,
            "max_tokens": 800
        }
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"].strip()
        else:
            print(f"DeepSeek API error {response.status_code}: {response.text}")
            return None
    except Exception as e:
        print(f"DeepSeek error: {e}")
        return None


print("LLM response functions defined (ChatGPT, Claude, DeepSeek)")